# OceanWatch · Consumo en Free Edition
Ejecutar en un workspace independiente. El proveedor institucional emite una activación de Open Sharing. Descarga el archivo `.share` una vez y súbelo a tu carpeta privada de Free. Escribe su ruta en el widget. También puedes instalar el perfil en Catalog Explorer con el nombre `oceanwatch_equipo8_open`. Este notebook vincula un catálogo de solo lectura y consulta las cuatro tablas directamente.

Para reutilizar la consola completa, importa `OceanWatch_Free.lvdash.json` en AI/BI Dashboards o usa el modo de publicación de la última celda. El warehouse y la carpeta usados aquí pertenecen a Free Edition.

In [ ]:
import json
from pathlib import Path
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.sharing import AuthenticationType

w = WorkspaceClient()
PROVEEDOR, CATALOGO_FREE, SHARE = "oceanwatch_equipo8_open", "oceanwatch_externo", "oceanwatch_eq8_oro"
dbutils.widgets.text("ruta_perfil", "", "Ruta privada al archivo .share")
dbutils.widgets.text("fecha", "2023-06-08", "Día AIS (UTC)")
dbutils.widgets.text("firma_proveedor", "", "JSON de control del proveedor (opcional)")
dbutils.widgets.text("warehouse_id", "", "SQL warehouse de Free Edition")
dbutils.widgets.text("ruta_dashboard", "", "Ruta a OceanWatch_Free.lvdash.json")
dbutils.widgets.dropdown("publicar", "no", ["no","si"], "Publicar tablero AI/BI")
try:
    proveedor = w.providers.get(PROVEEDOR)
except NotFound:
    ruta = dbutils.widgets.get("ruta_perfil").strip()
    if not ruta:
        raise ValueError("Indica la ruta al .share de activación o importa primero el proveedor con Catalog Explorer.")
    with Path(ruta).open(encoding="utf-8-sig") as f:
        perfil = json.load(f)
    if not (perfil.get("endpoint","").startswith("https://") and perfil.get("bearerToken")):
        raise ValueError("El perfil debe corresponder a Open Sharing con token.")
    proveedor = w.providers.create(name=PROVEEDOR, authentication_type=AuthenticationType.TOKEN,
                                   recipient_profile_str=json.dumps(perfil))
    del perfil
if proveedor.authentication_type != AuthenticationType.TOKEN:
    raise ValueError("El proveedor debe estar instalado mediante Open Sharing con token.")
if SHARE not in {s.name for s in w.providers.list_shares(PROVEEDOR)}:
    raise ValueError("El perfil no publica el share esperado.")
try:
    cat = w.catalogs.get(CATALOGO_FREE)
except NotFound:
    cat = w.catalogs.create(name=CATALOGO_FREE,provider_name=PROVEEDOR,share_name=SHARE)
if cat.provider_name != PROVEEDOR or cat.share_name != SHARE:
    raise ValueError("El catálogo de lectura existente no corresponde a este proveedor y share.")
spark.conf.set("spark.sql.session.timeZone","UTC")
tablas_free = [f"{CATALOGO_FREE}.oro.oceanwatch_gold_{n}" for n in
               ("trayectorias","presencia_puertos","trafico_celda_hora","anomalias")]
print("Catálogo externo",cat.name,"proveedor",cat.provider_name,"share",cat.share_name)
display(spark.sql(f"SHOW TABLES IN {CATALOGO_FREE}.oro"))

## Verificación independiente
El resultado identifica el workspace consumidor y los totales y huellas de cada tabla. Si pegas la firma de control del proveedor, se verifican también el día y la independencia de los workspaces. El token y el archivo de credenciales se mantienen fuera de las salidas.

In [ ]:
def firma_diaria(tablas, dia):
    import datetime
    from databricks.sdk import WorkspaceClient
    iso = datetime.date.fromisoformat(str(dia)).isoformat()
    resultado = {}
    for tabla in tablas:
        # Misma receta en ambos workspaces. Las cuatro huellas solo contienen agregados.
        cols = spark.table(tabla).columns
        campos = ",".join("`"+c.replace("`","``")+"`" for c in cols)
        f = spark.sql(f"""SELECT count(*) n, coalesce(sum(cast(xxhash64({campos}) as decimal(38,0))),0) h
                        FROM {tabla} WHERE fecha=DATE '{iso}'""").first()
        resultado[tabla.replace("`","").split(".")[-1]] = {"filas":int(f.n),"suma_hash":str(f.h)}
    return {"workspace":WorkspaceClient().config.host.rstrip("/"), "fecha":iso, "tablas":resultado}

In [ ]:
firma = firma_diaria(tablas_free, dbutils.widgets.get("fecha"))
if firma["tablas"]["oceanwatch_gold_trayectorias"]["filas"] == 0:
    raise ValueError("Elige un día ya procesado por el proveedor.")
print(json.dumps(firma,ensure_ascii=False))
texto = dbutils.widgets.get("firma_proveedor").strip()
if texto:
    esperado = json.loads(texto)
    if firma["workspace"] == esperado["workspace"]:
        raise ValueError("Usa un workspace Free independiente.")
    if firma["fecha"] != esperado["fecha"] or firma["tablas"] != esperado["tablas"]:
        raise ValueError("Día o huellas diferentes.")
    print("Las cuatro tablas coinciden entre los dos workspaces.")


## Publicación del tablero externo
Completa `warehouse_id` si hay varios almacenes y `ruta_dashboard` con la ruta al JSON del consumidor. Selecciona `publicar=si` y ejecuta la última celda. El filtro por día y las cuatro consultas operan sobre `oceanwatch_externo.oro`.

In [ ]:
def publicar_tablero(archivo, titulo, warehouse, copia=""):
    import json
    from pathlib import Path
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.dashboards import Dashboard

    w = WorkspaceClient()
    user = spark.sql("SELECT current_user()").first()[0]
    carpeta = f"/Users/{user}/OceanWatch"
    w.workspace.mkdirs(carpeta)
    caminos = [Path(copia)] if copia else []
    caminos += [Path.cwd().parent/"dashboards"/archivo, Path.cwd()/"dashboards"/archivo,
                Path.cwd()/archivo]
    ruta = next((p for p in caminos if p.is_file()), None)
    if ruta is None:
        raise FileNotFoundError(f"Indica en ruta_dashboard la ruta absoluta al archivo {archivo} del paquete.")
    definicion = json.loads(ruta.read_text(encoding="utf-8"))
    if not warehouse:
        candidatos = list(w.warehouses.list())
        if len(candidatos) != 1:
            raise ValueError("Selecciona un SQL warehouse accesible e indica su ID en warehouse_id.")
        warehouse = candidatos[0].id
    # Comprobar en el mismo warehouse que usará el lector del dashboard.
    from databricks.sdk.service import sql
    import datetime, time
    dia_widget = dbutils.widgets.get("fecha").strip()
    dia_sql = globals().get("fecha_dia", None) if dia_widget.upper()=="AUTO" else datetime.date.fromisoformat(dia_widget)
    if dia_sql is None:
        raise ValueError("Selecciona un día AIS explícito para publicar el tablero.")
    pruebas_sql = {}
    for ds in definicion["datasets"]:
        consulta = "SELECT * FROM (" + "".join(ds["queryLines"]) + ") ow LIMIT 1"
        st = w.statement_execution.execute_statement(consulta,warehouse,
            parameters=[sql.StatementParameterListItem(name="dia",type="DATE",value=dia_sql.isoformat())],
            wait_timeout="30s")
        limite = time.monotonic() + 300
        while st.status.state in {sql.StatementState.RUNNING,sql.StatementState.PENDING} and time.monotonic() < limite:
            time.sleep(3)
            st = w.statement_execution.get_statement(st.statement_id)
        if st.status.state != sql.StatementState.SUCCEEDED:
            detalle = st.status.error.message if st.status.error else st.status.state.value
            raise RuntimeError(f"Dataset {ds['name']}, sentencia {st.statement_id}: {detalle}")
        pruebas_sql[ds["name"]] = st.statement_id
    print("Consultas validadas en el warehouse:",pruebas_sql)
    existentes = [d for d in w.lakeview.list() if d.display_name == titulo and
                  ((d.parent_path or "").removeprefix("/Workspace") == carpeta or
                   str(d.path or "").removeprefix("/Workspace").rsplit("/",1)[0] == carpeta)]
    if len(existentes) > 1:
        raise ValueError(f"Hay varios dashboards con el nombre {titulo} en {carpeta}.")
    definicion = json.dumps(definicion, ensure_ascii=False)
    if existentes:
        previo = w.lakeview.get(existentes[0].dashboard_id)
        d = w.lakeview.update(previo.dashboard_id, Dashboard(display_name=titulo,
            etag=previo.etag, warehouse_id=warehouse, serialized_dashboard=definicion))
    else:
        d = w.lakeview.create(Dashboard(display_name=titulo, parent_path=carpeta, warehouse_id=warehouse,
                                         serialized_dashboard=definicion))
    w.lakeview.publish(d.dashboard_id, warehouse_id=warehouse, embed_credentials=False)
    w.lakeview.get_published(d.dashboard_id)
    print("Tablero publicado:", w.config.host.rstrip("/") + "/dashboardsv3/" + d.dashboard_id + "/published")
    return d.dashboard_id

In [ ]:
if dbutils.widgets.get("publicar") == "si":
    publicar_tablero("OceanWatch_Free.lvdash.json", "OceanWatch · Consumo externo",
                     dbutils.widgets.get("warehouse_id").strip(),dbutils.widgets.get("ruta_dashboard").strip())